# 6.6 — Vector Storage Compared

**Chapter 6, sections 6.1.1, 6.1.5 and 6.2.1.**

**The question this notebook answers:** a feature vector can be stored as a Python list, a NumPy
array, a SciPy sparse matrix, a `pyspark.ml.linalg` vector, a `pyspark.mllib.linalg` vector, or a
column of a DataFrame. They hold the same numbers. What does the choice cost when the vector is
used the way machine learning uses it — a dot product for a prediction, an *axpy* for a gradient
step, and a full pass over the data for a batch gradient?

The answer has two halves. The first is local: the container decides the cost of every arithmetic
operation on it. The second is distributed: **the ranking inside a Spark task is the same
ranking**, plus one more cost — where the vector is stored decides whether a pass over the data has
to cross the JVM/Python boundary.

Every timing below is **machine-dependent**. The ratios are the lesson, not the microseconds.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, sys, tempfile, timeit, time, pickle
import numpy as np
import scipy.sparse as sps
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-6.6")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

import scipy
print("Spark", spark.version, "| NumPy", np.__version__, "| SciPy", scipy.__version__)

Spark 4.2.0 | NumPy 2.5.2 | SciPy 1.18.1


---

## 1. One vector, five containers

A weight vector $\mathbf{w}$ and a feature vector $\mathbf{x}$ of $d = 10{,}000$ entries, drawn
once with a fixed seed, then stored five ways. The two Spark namespaces are both here because the
chapter's note box says a vector from one is not accepted where the other is required — so a
student is entitled to ask whether one is faster.

In [2]:
from pyspark.ml.linalg import Vectors as MLVectors
from pyspark.mllib.linalg import Vectors as MLlibVectors

d = 10_000
rng = np.random.default_rng(777)

w_np, x_np = rng.standard_normal(d), rng.standard_normal(d)      # NumPy ndarray
w_list, x_list = w_np.tolist(), x_np.tolist()                    # Python list
w_csr, x_csr = (sps.csr_matrix(v.reshape(1, -1)) for v in (w_np, x_np))  # SciPy, 1 x d
w_ml, x_ml = MLVectors.dense(w_np), MLVectors.dense(x_np)        # pyspark.ml
w_mllib, x_mllib = MLlibVectors.dense(w_np), MLlibVectors.dense(x_np)  # pyspark.mllib

containers = {
    "Python list":     (w_list, x_list),
    "NumPy ndarray":   (w_np, x_np),
    "SciPy csr_matrix": (w_csr, x_csr),
    "ml DenseVector":  (w_ml, x_ml),
    "mllib DenseVector": (w_mllib, x_mllib),
}
for name, (w, _) in containers.items():
    print(f"{name:<18} {type(w).__module__}.{type(w).__name__}")

Python list        builtins.list
NumPy ndarray      numpy.ndarray
SciPy csr_matrix   scipy.sparse._csr.csr_matrix
ml DenseVector     pyspark.ml.linalg.DenseVector
mllib DenseVector  pyspark.mllib.linalg.DenseVector


Same numbers, five types. As always in this course, "the same" is an assertion, not a sentence.

In [3]:
as_array = {
    "Python list": np.array(w_list),
    "NumPy ndarray": w_np,
    "SciPy csr_matrix": w_csr.toarray().ravel(),
    "ml DenseVector": w_ml.toArray(),
    "mllib DenseVector": w_mllib.toArray(),
}
for name, arr in as_array.items():
    assert np.array_equal(arr, w_np), name
print("all five hold the same", d, "values")

all five hold the same 10000 values


---

## 2. What each container occupies

A NumPy array is one contiguous block of `float64`, 8 bytes per entry. A Python list of floats is a
block of **pointers**, each pointing at a separately allocated `float` object of 24 bytes — so
four times the memory and, more importantly, four times the memory traffic every time the vector
is read.

Both Spark vector classes hold their numbers in a NumPy array, reachable as `.array`. They are
NumPy with a thin wrapper, and the wrapper is what the timings in the next section measure.

In [4]:
def nbytes(v):
    if isinstance(v, list):
        return sys.getsizeof(v) + sum(sys.getsizeof(e) for e in v)
    if sps.issparse(v):
        return v.data.nbytes + v.indices.nbytes + v.indptr.nbytes
    if hasattr(v, "array"):          # ml / mllib DenseVector
        return v.array.nbytes
    return v.nbytes                  # ndarray

print(f"{'container':<18} {'bytes':>9}  per entry")
for name, (w, _) in containers.items():
    b = nbytes(w)
    print(f"{name:<18} {b:>9,}  {b / d:6.1f}")

print()
print("ml    DenseVector.array is a", type(w_ml.array).__name__)
print("mllib DenseVector.array is a", type(w_mllib.array).__name__)

container              bytes  per entry
Python list          320,056    32.0
NumPy ndarray         80,000     8.0
SciPy csr_matrix     120,008    12.0
ml DenseVector        80,000     8.0
mllib DenseVector     80,000     8.0

ml    DenseVector.array is a ndarray
mllib DenseVector.array is a ndarray


The SciPy row is the odd one out: storing a **fully dense** vector in a sparse format costs the
values *plus* an index per value. Sparse storage is only a saving when most entries are zero,
which is section 5.

---

## 3. A prediction: the dot product

A linear model's prediction is $\hat{y} = \mathbf{w}^\top \mathbf{x}$. One dot product per
record, so its cost is the cost of inference.

`bench` takes the **best** of several repeats. Best-of is the right statistic for asking "how
fast can this operation run", because everything that makes a repeat slower — a garbage
collection, another process — is noise, not the operation.

In [5]:
def bench(fn, number=200, repeat=5):
    # microseconds per call, best of `repeat` batches of `number` calls
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number * 1e6

dot = {
    "Python list":       lambda: sum(a * b for a, b in zip(w_list, x_list)),
    "NumPy ndarray":     lambda: w_np @ x_np,
    "SciPy csr_matrix":  lambda: x_csr.dot(w_np),
    "ml DenseVector":    lambda: x_ml.dot(w_ml),
    "mllib DenseVector": lambda: x_mllib.dot(w_mllib),
}
base = bench(dot["NumPy ndarray"])
print(f"{'dot product, d = 10,000':<18} {'us/call':>9}  vs NumPy")
for name, fn in dot.items():
    t = bench(fn, number=20 if name == "Python list" else 200)
    print(f"{name:<18} {t:>9.2f}  {t / base:6.1f}x")

dot product, d = 10,000   us/call  vs NumPy
Python list           203.13   220.0x
NumPy ndarray           0.85     0.9x
SciPy csr_matrix        9.61    10.4x
ml DenseVector          1.09     1.2x
mllib DenseVector       1.06     1.2x


Three things to read off this table.

* **The Python list is two orders of magnitude slower.** Every multiply is an interpreted
  bytecode step on a boxed float. This is the reason chapter 6 opens with NumPy at all: inside a
  task, the arithmetic is NumPy's or it is slow.
* **The two Spark vectors cost NumPy plus a fraction of a microsecond.** `DenseVector.dot`
  checks the type of its argument and calls `np.dot`. `ml` and `mllib` are the same
  implementation in two namespaces; neither is faster, and the choice between them is the
  namespace question of section 6.2, not a performance question.
* **SciPy is slower than NumPy here** because the vector is dense. The sparse machinery — walk the
  index array, gather — is pure overhead when every index is present.

---

## 4. A gradient step: $\mathbf{w} \leftarrow \mathbf{w} - \eta\,\mathbf{g}$

The other half of training. In BLAS vocabulary this is an *axpy* — $a\mathbf{x} + \mathbf{y}$ —
and it produces a new vector rather than a scalar, so the cost of allocating the result is part of
the measurement. The Spark vectors support arithmetic operators directly and return a vector of
their own type. SciPy is left out: a sparse format for a dense weight vector has already lost
section 2.

In [6]:
eta = 0.01
g_list, g_np, g_ml, g_mllib = x_list, x_np, x_ml, x_mllib   # the gradient, in each container

axpy = {
    "Python list":       lambda: [a - eta * b for a, b in zip(w_list, g_list)],
    "NumPy ndarray":     lambda: w_np - eta * g_np,
    "ml DenseVector":    lambda: w_ml - eta * g_ml,
    "mllib DenseVector": lambda: w_mllib - eta * g_mllib,
}
base = bench(axpy["NumPy ndarray"])
print(f"{'axpy, d = 10,000':<18} {'us/call':>9}  vs NumPy   result type")
for name, fn in axpy.items():
    t = bench(fn, number=20 if name == "Python list" else 200)
    print(f"{name:<18} {t:>9.2f}  {t / base:6.1f}x   {type(fn()).__name__}")

axpy, d = 10,000     us/call  vs NumPy   result type
Python list           180.53    51.8x   list
NumPy ndarray           3.81     1.1x   ndarray
ml DenseVector          3.93     1.1x   DenseVector
mllib DenseVector       3.88     1.1x   DenseVector


Same ranking, and the Spark vectors return a `DenseVector`, so a gradient loop written against
them stays in their type. The wrapper's cost is again a fraction of a microsecond per operation —
invisible on a 10,000-element vector, and the reason the *only* place it could ever matter is a
loop over millions of **tiny** vectors, which is what a per-record `map` in Spark is. Section 7
measures exactly that.

---

## 5. When sparse wins: the same product at 1 % density

Now $d = 100{,}000$ with **1,000 non-zeros** — the shape of a one-hot or bag-of-words feature
vector. The dense containers store all 100,000 entries; the sparse ones store 1,000 indices and
1,000 values. Notebook 6.3 computed what that saves in bytes; this section measures what it
costs in time.

Two sparse dot products are timed for the Spark vector: against a **dense** `w` (the
prediction case — weights are dense) and against **another sparse vector** (which is the slow
case, and worth knowing about).

In [7]:
D, NNZ = 100_000, 1_000
idx = np.sort(rng.choice(D, NNZ, replace=False))
vals = rng.standard_normal(NNZ)
wD = rng.standard_normal(D)                       # dense weights

xs_np = np.zeros(D); xs_np[idx] = vals            # sparse data, dense container
xs_csr = sps.csr_matrix((vals, idx, [0, NNZ]), shape=(1, D))
xs_ml = MLVectors.sparse(D, idx.tolist(), vals.tolist())
xs_mllib = MLlibVectors.sparse(D, idx.tolist(), vals.tolist())
wD_ml, wD_mllib = MLVectors.dense(wD), MLlibVectors.dense(wD)
idx_list, vals_list, wD_list = idx.tolist(), vals.tolist(), wD.tolist()

sparse_dot = {
    "list over non-zeros":         lambda: sum(wD_list[i] * v for i, v in zip(idx_list, vals_list)),
    "NumPy dense . dense":         lambda: xs_np @ wD,
    "SciPy csr . dense":           lambda: xs_csr.dot(wD),
    "ml Sparse . ndarray":         lambda: xs_ml.dot(wD),
    "ml Sparse . DenseVector":     lambda: xs_ml.dot(wD_ml),
    "mllib Sparse . DenseVector":  lambda: xs_mllib.dot(wD_mllib),
    "ml Sparse . Sparse":          lambda: xs_ml.dot(xs_ml),
}
expected = xs_np @ wD
base = bench(sparse_dot["NumPy dense . dense"])
print(f"{'d = 100,000, nnz = 1,000':<27} {'us/call':>8}  vs NumPy dense")
for name, fn in sparse_dot.items():
    if "Sparse . Sparse" not in name:
        assert np.isclose(fn(), expected), name
    t = bench(fn, number=50)
    print(f"{name:<27} {t:>8.2f}  {t / base:6.1f}x")

print()
print(f"bytes  dense  : {xs_np.nbytes:>9,}")
print(f"bytes  sparse : {xs_ml.indices.nbytes + xs_ml.values.nbytes:>9,}"
      f"   ({xs_np.nbytes / (xs_ml.indices.nbytes + xs_ml.values.nbytes):.0f}x smaller)")

d = 100,000, nnz = 1,000     us/call  vs NumPy dense
list over non-zeros            25.00     9.6x
NumPy dense . dense             2.53     1.0x
SciPy csr . dense               2.09     0.8x
ml Sparse . ndarray             2.26     0.9x
ml Sparse . DenseVector         2.31     0.9x
mllib Sparse . DenseVector      2.32     0.9x
ml Sparse . Sparse             31.46    12.0x

bytes  dense  :   800,000
bytes  sparse :    12,000   (67x smaller)


The honest reading: at this size the sparse dot product against dense weights is **about as fast
as NumPy's dense one**, not dramatically faster. NumPy streams 800 KB of dense data through a
BLAS kernel in a few microseconds; the sparse route does a 1,000-element gather and a
1,000-element dot, which is also a few microseconds. What sparse buys at 1 % density is the
**67× smaller footprint**, and the next section shows why that footprint matters more in Spark
than on a laptop.

The Python loop over the non-zeros is the one case where a list is competitive, because it does
1,000 operations instead of 100,000 — sparsity helps every container.

The **sparse-times-sparse** product is the slow case: the two index arrays have to be intersected
before anything is multiplied. Keep the weight vector dense.

---

## 6. The cost of moving a record

Nothing in Spark's Python API stays where it was created. A record in an RDD is **pickled** to be
cached, shuffled, or handed from the JVM to a Python worker and back. So the container's third
cost, after memory and arithmetic, is how quickly it serializes and how many bytes it becomes.

In [8]:
def pickled(obj):
    return pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)

movers = {
    "Python list (d=10,000)":      w_list,
    "NumPy ndarray (d=10,000)":    w_np,
    "ml DenseVector (d=10,000)":   w_ml,
    "SciPy csr (d=10,000, dense)": w_csr,
    "NumPy dense (d=100,000, 1% nnz)": xs_np,
    "ml SparseVector (d=100,000, 1% nnz)": xs_ml,
}
print(f"{'record':<36} {'bytes':>9} {'dumps us':>9} {'loads us':>9}")
for name, obj in movers.items():
    blob = pickled(obj)
    t_dump = bench(lambda: pickled(obj), number=50)
    t_load = bench(lambda: pickle.loads(blob), number=50)
    print(f"{name:<36} {len(blob):>9,} {t_dump:>9.1f} {t_load:>9.1f}")

record                                   bytes  dumps us  loads us
Python list (d=10,000)                  90,043      51.5      83.4
NumPy ndarray (d=10,000)                80,137       2.7       1.9
ml DenseVector (d=10,000)               80,067       4.1       1.4
SciPy csr (d=10,000, dense)            120,328       5.6       4.0
NumPy dense (d=100,000, 1% nnz)        800,139      11.2      11.0
ml SparseVector (d=100,000, 1% nnz)     12,070       0.9      68.0


A NumPy array pickles as one header plus one memory copy. A list of floats pickles **one float at
a time**, an order of magnitude slower, and lands as a list of pointers again on the other side.
The sparse vector is the largest win in the table: 67× fewer bytes to move per record, at every
boundary the record crosses.

Its `loads` column is the surprise, and it is worth knowing where it comes from. `SparseVector`
pickles as three raw byte strings, which is why `dumps` is fastest of all; but unpickling calls
the constructor, and the constructor **re-checks that the indices are sorted in a Python `for`
loop** — so receiving a sparse vector costs Python time per non-zero. Read the instrument
carefully: the bytes moved are 67× fewer, the time to rebuild the object is not.

---

## 7. The same ranking inside a Spark task

A batch gradient for least squares over $n$ records,
$\mathbf{g} = \sum_i (\mathbf{w}^\top\mathbf{x}_i - y_i)\,\mathbf{x}_i$, is one `map` (the
per-record term) and one `treeAggregate` (the sum) — the distributed-iteration shape of section
6.5. The per-record term is a dot product and a scaled copy: sections 3 and 4, run $n$ times.

$n = 100{,}000$ records of $d = 200$ features, generated **inside the tasks** from a seed per
partition, so nothing large crosses from the driver. Every variant below stores the same rows in a
different container.

**Fairness rule.** Each variant is cached and materialized with `count()` before it is timed, and
unpersisted after, so no variant is timed against another's saved work and only one copy of the
data is resident at a time. The time it takes to cache is reported too — that is the cost of
section 6, paid once per record.

In [9]:
from pyspark.ml.linalg import Vectors

n, d, P = 100_000, 200, 4
w_true = np.random.default_rng(1).standard_normal(d)

def make_rows(pid, _rows):
    r = np.random.default_rng(pid)
    for _ in _rows:
        x = r.standard_normal(d)
        yield x, float(x @ w_true) + 0.1 * r.standard_normal()

rows = sc.parallelize(range(n), P).mapPartitionsWithIndex(make_rows)   # (ndarray, float)
w = np.zeros(d)                                                         # the current model

def timed(label, fn, reps=3):
    best, out = float("inf"), None
    for _ in range(reps):
        t0 = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t0)
    results[label] = best
    print(f"{label:<28} {best:6.2f} s   (best of {reps})")
    return out

def materialize(rdd_or_df):
    t0 = time.perf_counter(); rdd_or_df.cache().count()
    return time.perf_counter() - t0

results, cache_time, grads = {}, {}, {}
print(f"{n:,} rows x {d} features in {P} partitions")

100,000 rows x 200 features in 4 partitions


**Variant A — a Python list per row.** The arithmetic is section 3's list comprehension, and the
`treeAggregate` has to add lists too.

In [10]:
rows_list = rows.map(lambda r: (r[0].tolist(), r[1]))
cache_time["Python list rows"] = materialize(rows_list)
w_l = w.tolist()

def term_list(row):
    x, y = row
    e = sum(a * b for a, b in zip(w_l, x)) - y
    return [e * a for a in x]

add_lists = lambda a, b: [p + q for p, q in zip(a, b)]
grads["Python list rows"] = timed("Python list rows",
    lambda: rows_list.map(term_list).treeAggregate([0.0] * d, add_lists, add_lists))
_ = rows_list.unpersist()

Python list rows               0.37 s   (best of 3)


**Variant B — a NumPy array per row.** The same expression, vectorized.

In [11]:
cache_time["NumPy rows"] = materialize(rows)

def term_np(row):
    x, y = row
    return (x @ w - y) * x

grads["NumPy rows"] = timed("NumPy rows",
    lambda: rows.map(term_np).treeAggregate(np.zeros(d), np.add, np.add))
_ = rows.unpersist()

NumPy rows                     0.06 s   (best of 3)


**Variant C — a `pyspark.ml` `DenseVector` per row.** The wrapper from section 3, now paid
100,000 times. This is the container a `features` column holds, so it is the one that matters.

In [12]:
rows_ml = rows.map(lambda r: (Vectors.dense(r[0]), r[1]))
cache_time["ml DenseVector rows"] = materialize(rows_ml)
w_ml = Vectors.dense(w)

def term_ml(row):
    x, y = row
    return (x.dot(w_ml) - y) * x            # DenseVector in, DenseVector out

zero_ml = Vectors.dense(np.zeros(d))
grads["ml DenseVector rows"] = timed("ml DenseVector rows",
    lambda: rows_ml.map(term_ml).treeAggregate(zero_ml, lambda a, b: a + b, lambda a, b: a + b))
_ = rows_ml.unpersist()

ml DenseVector rows            0.08 s   (best of 3)


**Variant D — a `features` column of a DataFrame.** The same `DenseVector`s, but now the
DataFrame is cached **on the JVM side**, in Spark's own columnar format. Running a Python
function over it means `df.rdd`, and `df.rdd` means every row is converted to a Python `Row` and
its vector unpickled — on **every** pass, because the cached thing is the JVM DataFrame, not the
Python objects.

This is what section 6.1.5 means by a Python function on a column being "on the wrong side of the
JVM/Python boundary". The instrument here measures that crossing plus the arithmetic; the
arithmetic is variant C.

In [13]:
df = spark.createDataFrame(rows_ml, ["features", "label"])
cache_time["DataFrame vector column"] = materialize(df)

def term_row(row):
    x = row["features"]
    return (x.dot(w_ml) - row["label"]) * x

grads["DataFrame vector column"] = timed("DataFrame vector column",
    lambda: df.rdd.map(term_row).treeAggregate(zero_ml, lambda a, b: a + b, lambda a, b: a + b))
_ = df.unpersist()

DataFrame vector column        0.43 s   (best of 3)


Four containers, one gradient. First the assertion that they agree, then the table.

In [14]:
ref = grads["NumPy rows"]
assert np.allclose(np.array(grads["Python list rows"]), ref)
assert np.allclose(grads["ml DenseVector rows"].toArray(), ref)
assert np.allclose(grads["DataFrame vector column"].toArray(), ref)
print("all four gradients agree\n")

base = results["NumPy rows"]
print(f"{'rows stored as':<26} {'cache s':>8} {'gradient s':>11} {'vs NumPy':>9} {'us/row':>8}")
for name in results:
    print(f"{name:<26} {cache_time[name]:>8.2f} {results[name]:>11.2f} "
          f"{results[name] / base:>8.1f}x {results[name] / n * 1e6:>8.1f}")

all four gradients agree

rows stored as              cache s  gradient s  vs NumPy   us/row
Python list rows               1.00        0.37      5.7x      3.7
NumPy rows                     0.14        0.06      1.0x      0.6
ml DenseVector rows            0.48        0.08      1.3x      0.8
DataFrame vector column        1.09        0.43      6.6x      4.3


The local ranking survives the trip to the cluster. Lists are several times slower than NumPy
per row — less than the 200× of section 3, because at $d = 200$ Spark's own per-record overhead
(iterating, pickling, scheduling) is a larger share of each row's cost, and that overhead is the
same for every container. The `DenseVector` wrapper costs a fraction of a microsecond per row,
which is what sections 3 and 4 predicted.

The DataFrame column is the outlier, and not because of the vector: because of **where it is
stored**. The JVM-side cache is the right place for a DataFrame that Catalyst will process, and
the wrong place for one a Python function will process on every iteration. The estimators in
`pyspark.ml`, chapter 7, resolve this by running the whole loop on the JVM side, so that the
vectors never cross at all.

---

## Conclusion

**The container a vector is stored in decides three costs — memory, arithmetic, and movement —
and the ranking is the same on the driver and inside a task.**

* A **Python list** of floats is 4× the memory of a NumPy array, ~200× slower to multiply, and an
  order of magnitude slower to pickle. Inside a Spark task it is the same object, with the same
  costs. There is no scale at which it is the right container for numbers.
* A **NumPy array** is the floor. Every faster option in this course is NumPy running somewhere
  else.
* **`pyspark.ml` and `pyspark.mllib` vectors are NumPy arrays with a wrapper**, costing a fraction
  of a microsecond per operation and returning their own type. They are the same implementation
  in two namespaces; choosing between them is section 6.2's namespace question, not a performance
  question.
* **Sparse** vectors — SciPy's or Spark's — win on **footprint** at low density (67× at 1 %), and
  that footprint is paid at every boundary a record crosses in Spark. On arithmetic they roughly
  tie with dense NumPy against a dense weight vector, and lose badly sparse-against-sparse.
* A **DataFrame column** stores the vector on the JVM side. That is where an estimator wants it
  and where a Python `map` does not; the crossing, not the vector, is what a per-iteration
  `df.rdd` pays.

In [15]:
spark.stop()